# 09 — v6: sibling stage 2, set selection, France fixes, memory-safe

Full run of `src/pipeline.py` on Kaggle's 2x T4 (`notebooks/kaggle/push.py 09_v6`):

1. Stage 1 as v3 (key blocking + bi-encoder dense candidates + cross-encoder + LightGBM), out-of-fold on fit rows.
2. **Stage 2:** sibling features (how a record compares with the records already assigned to the entity, and how that
   support compares across the record's candidates) + a second LightGBM on uncertain pairs.
3. **Decision:** stage-1 threshold, stage-2 threshold or per-entity expected-F0.5 set selection — whichever validates best.
4. **France:** French addresses no longer get US/India aliases (`st` → `saint`, not `street`); self-training before stage 2.
5. **Memory:** candidates as int32 positions (no id strings), blocking keys hashed and generated per chunk, each stage in
   its own process.

Outputs: `output/` (submission files); `artifacts/` — stage-1-only and no-France-self-training variants, cached validation
and test scores for re-tuning without re-running.

In [ ]:
import sys, json
from pathlib import Path
sys.path.insert(0, str(Path.cwd().resolve().parent / "src"))
import pandas as pd
import matplotlib.pyplot as plt
from config import ARTIFACT_DIR, DATA_DIR, OUTPUT_DIR, ON_KAGGLE
from pipeline import RunConfig, run
pd.set_option("display.width", 200)
print(f"data {DATA_DIR}\noutput {OUTPUT_DIR}\nartifacts {ARTIFACT_DIR}\non Kaggle: {ON_KAGGLE}")
cfg = RunConfig(use_neural=True, stage2=True, france_self_train=True)
cfg

In [ ]:
report = run(cfg, OUTPUT_DIR, ARTIFACT_DIR)
print("decision method:", report["method"], "| neural stages:", report["neural"])

## Validation (official macro F0.5), per decision method

In [ ]:
pd.DataFrame(report["validation"]).T

In [ ]:
pd.DataFrame(report["blocking"]).T

In [ ]:
print(json.dumps(report["diagnostics"], indent=2))

In [ ]:
pd.Series(report["top_features"]).to_frame("gain (stage 1)")

## Test output

In [ ]:
print(json.dumps(report["test"], indent=2, default=str))
for p in [OUTPUT_DIR / "matching_results.tsv", OUTPUT_DIR / "candidate_pairs.tsv"] + sorted(ARTIFACT_DIR.glob("matching_results_*.tsv")):
    print(f"{p.name}: {p.stat().st_size / 1e6:.1f} MB")